# Phase 2R-D：pure DNN 最终判别

只读取冻结结果，不重新求解 MPC。

In [1]:
from pathlib import Path
import json, pandas as pd
from IPython.display import display
ROOT=Path.cwd(); OUT=ROOT/'outputs'/'phase2rd_final_discrimination'
if not OUT.exists(): ROOT=ROOT.parent; OUT=ROOT/'outputs'/'phase2rd_final_discrimination'
m=json.loads((OUT/'metrics.json').read_text(encoding='utf-8')); d1=pd.read_csv(OUT/'multistart_state_summary.csv'); v=pd.read_csv(OUT/'local_variance_sensitivity.csv'); print(m['decision'])

{'continue_pure_dnn': False, 'conclusion': '停止 pure DNN 直接替代路线', 'reason': '相同完整状态下仍存在不可忽略的近最优第一动作多值性。'}


## D1：完全相同状态的多起点求解

In [2]:
display(pd.DataFrame([m['d1_multistart']])); display(d1.sort_values('near_optimal_first_action_range_a',ascending=False).head(10))

,state_count,warm_starts_per_state,near_optimal_multivalued_state_count,near_optimal_multivalued_fraction,warm_start_sensitive_state_count,all_starts_feasible_state_count,maximum_near_optimal_first_action_range_a,control_law_multivalued
0,100,15,57,0.57,58,100,0.365852,True


,state_id,successful_feasible_count,feasible_count,fallback_count,optimizer_success_count,first_action_range_a,near_optimal_first_action_range_a,objective_range,near_optimal_solution_count,active_mode_count,status_count,near_optimal_multivalued,warm_start_sensitive
36,state_036,15,15,0,15,0.365852,0.365852,0.000094,15,1,1,True,True
94,state_094,15,15,0,15,0.314553,0.314553,0.000022,15,1,1,True,True
61,state_061,15,15,0,15,0.285122,0.285122,0.000053,15,1,1,True,True
18,state_018,15,15,0,15,0.275544,0.275544,0.000119,15,1,1,True,True
29,state_029,15,15,0,15,0.273253,0.273253,0.000069,15,1,1,True,True
54,state_054,15,15,0,15,0.269820,0.269820,0.000058,15,1,1,True,True
40,state_040,15,15,0,15,0.247843,0.247843,0.000075,15,1,1,True,True
93,state_093,15,15,0,15,0.235997,0.235997,0.000033,15,1,1,True,True
19,state_019,15,15,0,15,0.231756,0.231756,0.000058,15,1,1,True,True
47,state_047,15,15,0,15,0.228418,0.228418,0.000035,15,1,1,True,True


## D2/D3：完整序列与 K/模式敏感性

In [3]:
view=v[(v.mode_group=='all') & (v.auditable==True)][['neighbor_count','feature_set','mean_local_standard_deviation_a','nearest_neighbor_label_difference_p95_a','locally_sufficient']]; display(view)

,neighbor_count,feature_set,mean_local_standard_deviation_a,nearest_neighbor_label_difference_p95_a,locally_sufficient
0,5,five_state,0.326824,1.020778,False
1,5,five_state_plus_summary,0.270853,0.809917,False
2,5,five_state_plus_full_previous_sequence,0.244525,0.561324,False
3,10,five_state,0.402640,1.020778,False
4,10,five_state_plus_summary,0.332078,0.809917,False
5,10,five_state_plus_full_previous_sequence,0.298181,0.561324,False
6,25,five_state,0.502610,1.020778,False
7,25,five_state_plus_summary,0.420474,0.809917,False
8,25,five_state_plus_full_previous_sequence,0.369936,0.561324,False
9,50,five_state,0.579415,1.020778,False


In [4]:
modes=v[(v.mode_group!='all') & (v.auditable==True) & (v.feature_set=='five_state_plus_full_previous_sequence')]; display(modes[['mode_group','sample_count','neighbor_count','mean_local_standard_deviation_a','nearest_neighbor_label_difference_p95_a','locally_sufficient']])

,mode_group,sample_count,neighbor_count,mean_local_standard_deviation_a,nearest_neighbor_label_difference_p95_a,locally_sufficient
26,voltage,288,5,0.249551,1.028559,False
29,voltage,288,10,0.353588,1.028559,False
32,voltage,288,25,0.464042,1.028559,False
35,voltage,288,50,0.564311,1.028559,False
38,temperature,1144,5,0.174190,0.365128,True
41,temperature,1144,10,0.205764,0.365128,True
44,temperature,1144,25,0.244589,0.365128,True
47,temperature,1144,50,0.279906,0.365128,False
62,voltage+temperature,94,5,0.230647,0.326942,True
65,voltage+temperature,94,10,0.326200,0.326942,False


## 最终判别

In [5]:
display(pd.DataFrame([m['decision']])); assert m['status']=='completed'

,continue_pure_dnn,conclusion,reason
0,False,停止 pure DNN 直接替代路线,相同完整状态下仍存在不可忽略的近最优第一动作多值性。
